In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
from pprint import pprint

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
# 簡単なPythonの関数をツールとして使用。
import json

# 検索結果を返す関数の作成
# keyword：検索キーワード
# when：日付や時間の範囲。デフォルトは "1"
# unit：日付・時間範囲の単位。デフォルトは月（"m"）


def get_search_result(keyword, when="1", unit="m"):
    result = {
        "result": [
            {"title": "1914年（大正3年）中央停車場改め「東京駅」として営業を開始しました"},
            {"title": "1929年（昭和4年）八重洲口が開設されました"},
            {"title": "2014年（平成26年）東京駅開業100周年を迎えました"},
        ]
    }

    return json.dumps(result)
    # result 辞書を json.dumps でJSON形式に変換し、文字列として返す

In [ ]:
# get_search_result関数をツールとして利用するための
# 「ツール定義」を作成

from openai.types.chat import ChatCompletionToolParam

# ツール定義
tools=[
    ChatCompletionToolParam({
        "type": "function",
        "function": {
            "name": "get_search_result",
            "description": "指定したキーワードの検索結果を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "キーワード",
                    },
                    "when": {
                        "type": "number",
                        "description": "日付や時間の範囲"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["d", "h", "y"]
                    },
                },
                "required": ["keyword"],
            },
        },
    })
]

In [ ]:
# 言語モデルが直接回答できる質問　--------

question = "日本の首都はどこにありますか？"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": question},
    ],
    tools=tools,
    tool_choice="auto", # オートにしてモデルにツールは任せている
)

# レスポンスの表示
pprint(vars(response))

{'_request_id': 'req_27361cdb5ff34656ac50a35f0cb12e64',
 'choices': [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='日本の首都は東京都（とうきょうと）にあります。東京都は日本の政治、経済、文化の中心地であり、国内外の多くの機関や企業が集まっています。', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))],
 'created': 1767080454,
 'id': 'chatcmpl-CsOda2DlMFjgimsJdTq2udSDzuUmU',
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_c4585b5b9c',
 'usage': CompletionUsage(completion_tokens=47, prompt_tokens=91, total_tokens=138, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))}


finish_reason='stop'（正常終了）。首都も正しく答えている。

In [12]:
# ツール呼出が必要な質問　--------
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": question},
    ],
    tools=tools,
    tool_choice="auto", # オートにしてモデルにツールは任せている
)

# レスポンスの表示
pprint(vars(response))

{'_request_id': 'req_e1c9298cef95496d8ac4e6f259d03f84',
 'choices': [Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_6OvjmIrrj99WLCSR9jjL0LCL', function=Function(arguments='{"keyword":"東京駅 イベント","when":30,"unit":"d"}', name='get_search_result'), type='function')], annotations=[]))],
 'created': 1767087862,
 'id': 'chatcmpl-CsQZ4zniKndFprI4aKRvK8tyODFxk',
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_c4585b5b9c',
 'usage': CompletionUsage(completion_tokens=26, prompt_tokens=97, total_tokens=123, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))}


今度はfinish_reason='tool_calls'（終了理由がツール呼出）。

言語モデルが「ツール呼出」が適切と判断して、コードを返したことがわかる↓
function=Function(arguments='{"keyword":"東京駅 イベント",
"when":30,"unit":"d"}', name='get_search_result')

次は帰ってきたコードを使って、質問と一緒にGPTモデルへ渡す

In [14]:
# モデルがツール呼出と判断した
if response.choices[0].finish_reason == 'tool_calls':
    # 言語モデルの回答からツールを取得
    tool = response.choices[0].message.tool_calls[0]

    # 関数名の取得
    function_name = tool.function.name
    print(f"関数名：{function_name}")

    # 引数の取得
    arguments = json.loads(tool.function.arguments)
    print(f"引数：{arguments}")

    # 関数の実行
    function_response = globals()[function_name](**arguments)
    print("関数の実行結果：")
    pprint(json.loads(function_response))

    # 関数の実行結果をmessagesに加えて言語モデルへ送信
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )

    # 言語モデルからの回答を出力
    print("言語モデルからの回答：")
    print(response_after_tool_call.choices[0].message.content.strip())

else:
    # 関数呼び出しでなければ、単に言語モデルからの回答を出力
    print("言語モデルからの回答：")
    print(response.choices[0].message.content.strip())

関数名：get_search_result
引数：{'keyword': '東京駅 イベント', 'when': 30, 'unit': 'd'}
関数の実行結果：
{'result': [{'title': '1914年（大正3年）中央停車場改め「東京駅」として営業を開始しました'},
            {'title': '1929年（昭和4年）八重洲口が開設されました'},
            {'title': '2014年（平成26年）東京駅開業100周年を迎えました'}]}
言語モデルからの回答：
最近1ヶ月以内の東京駅に関するイベント情報は以下の通りです：

1. **1914年（大正3年）中部停車場改め「東京駅」として営業を開始しました** - 東京駅が開業した年月についてのイベントです。
2. **1929年（昭和4年）八重洲口が開設されました** - 東京駅の八重洲口の開設に関するイベント。
3. **2014年（平成26年）東京駅開業100周年を迎えました** - 東京駅開業100周年を祝ったイベント。

これらのイベントは東京駅の歴史に関連しており、特定の新しいイベントの情報は含まれていませんでした。詳細な最新情報については、公式のイベントサイトや東京駅の情報をチェックすることをおすすめします。
